# LeetCode 724 — Find Pivot Index bằng Python

**Link đề:** https://leetcode.com/problems/find-pivot-index/
**Độ khó:** Easy · **Chủ đề:** Array, Prefix Sum

```python
class Solution:
    def pivotIndex(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đúng định nghĩa **pivot index**: tổng bên trái *nghiêm ngặt* bằng tổng bên phải *nghiêm ngặt*.
2. Ôn `list`, indexing, slicing `nums[a:b]`, hàm `sum()`, `enumerate`, `range`.
3. Viết cách trực giác (Brute Force `O(n²)`), nhận ra **phần việc bị lặp lại**.
4. Học kỹ thuật **Prefix Sum** (tổng tiền tố) theo 2 mức: mảng tiền tố `O(n)` bộ nhớ → chỉ một biến `left` `O(1)` bộ nhớ.
5. Trace từng bước, chứng minh đúng bằng invariant, test đối chiếu, làm bài tập biến thể.

## 1. Đề bài

Cho mảng số nguyên `nums`. **Pivot index** là chỉ số `i` sao cho:

```text
tổng các phần tử bên TRÁI i  (nums[0] + ... + nums[i-1])
  ==
tổng các phần tử bên PHẢI i  (nums[i+1] + ... + nums[n-1])
```

- Phần tử `nums[i]` **không** thuộc bên nào.
- Nếu `i = 0` thì bên trái rỗng → tổng trái = `0`. Tương tự `i = n-1` thì tổng phải = `0`.
- Trả về pivot **trái nhất** (chỉ số nhỏ nhất). Không có → trả `-1`.

| `nums` | Kết quả | Giải thích |
|---|:---:|---|
| `[1,7,3,6,5,6]` | `3` | trái `1+7+3 = 11`, phải `5+6 = 11` |
| `[1,2,3]` | `-1` | không chỉ số nào cân bằng |
| `[2,1,-1]` | `0` | trái rỗng = 0, phải `1 + (-1) = 0` |

**Constraints:** `1 <= len(nums) <= 10^4`, `-1000 <= nums[i] <= 1000` (có **số âm**!).

In [1]:
nums = [1, 7, 3, 6, 5, 6]
i = 3
left_part = nums[:i]
right_part = nums[i + 1:]
print("nums[i] =", nums[i])
print("bên trái :", left_part, "-> tổng", sum(left_part))
print("bên phải :", right_part, "-> tổng", sum(right_part))

nums[i] = 6
bên trái : [1, 7, 3] -> tổng 11
bên phải : [5, 6] -> tổng 11


### Tự kiểm tra bằng tay trước khi code

Với `nums = [2, 1, -1]`:

| i | bên trái | tổng trái | bên phải | tổng phải | Cân bằng? |
|---:|---|---:|---|---:|:---:|
| 0 | `[]` | 0 | `[1, -1]` | 0 | ✅ → trả 0 |
| 1 | `[2]` | 2 | `[-1]` | -1 | ❌ |
| 2 | `[2, 1]` | 3 | `[]` | 0 | ❌ |

Vì cần pivot **trái nhất**, gặp `i = 0` là trả ngay, không cần xét tiếp.

## 2. Nền tảng Python cần dùng

### 2.1. `list` và indexing

`nums[i]` đọc phần tử thứ `i` (đếm từ 0). `len(nums)` là số phần tử. Chỉ số hợp lệ: `0 .. len(nums) - 1`.

In [2]:
nums = [1, 7, 3, 6, 5, 6]
print(len(nums))     # 6
print(nums[0])       # 1  (phần tử đầu)
print(nums[5])       # 6  (phần tử cuối)
print(nums[-1])      # 6  (chỉ số âm: đếm từ cuối)

6
1
6
6


### 2.2. Slicing `nums[a:b]`

`nums[a:b]` tạo **list mới** gồm các phần tử từ chỉ số `a` đến `b - 1` (không lấy `b`).

- `nums[:i]` = các phần tử **trước** `i` → đúng "bên trái".
- `nums[i+1:]` = các phần tử **sau** `i` → đúng "bên phải".
- Slice vượt biên **không lỗi**, chỉ cho list rỗng: `nums[10:]` → `[]`.

In [3]:
nums = [1, 7, 3, 6, 5, 6]
print(nums[:3])      # [1, 7, 3]
print(nums[4:])      # [5, 6]
print(nums[:0])      # []  -> bên trái của i = 0
print(nums[6:])      # []  -> bên phải của i = 5

[1, 7, 3]
[5, 6]
[]
[]


### 2.3. `sum()` và tổng của list rỗng

`sum(list)` cộng mọi phần tử. Quan trọng: `sum([]) == 0` — khớp với quy ước của đề "bên rỗng có tổng 0", nên ta **không cần xử lý riêng** hai đầu mảng.

In [4]:
print(sum([1, 7, 3]))   # 11
print(sum([]))          # 0
print(sum([1, -1]))     # 0  (số âm cộng bình thường)

11
0
0


### 2.4. `enumerate` — vừa có chỉ số vừa có giá trị

`for i, x in enumerate(nums)` cho từng cặp `(i, nums[i])`. Tiện hơn `for i in range(len(nums))` rồi lại viết `nums[i]`.

In [5]:
for i, x in enumerate([4, 9, 2]):
    print("i =", i, "| x =", x)

i = 0 | x = 4
i = 1 | x = 9
i = 2 | x = 2


## 3. Cách 1 — Trực giác / Brute Force

Đọc đề ra code ngay: với mỗi `i`, cắt bên trái, cắt bên phải, cộng từng bên rồi so sánh. Duyệt `i` từ nhỏ đến lớn nên cái đầu tiên tìm thấy là trái nhất.

In [6]:
def pivot_bruteforce(nums: list[int]) -> int:
    for i in range(len(nums)):
        if sum(nums[:i]) == sum(nums[i + 1:]):
            return i
    return -1

print(pivot_bruteforce([1, 7, 3, 6, 5, 6]))  # 3
print(pivot_bruteforce([1, 2, 3]))           # -1
print(pivot_bruteforce([2, 1, -1]))          # 0

3
-1
0


### Chi phí của Brute Force

Với mỗi `i`, hai lời gọi `sum` cộng tổng cộng `n - 1` phần tử (cộng thêm chi phí tạo 2 slice). Có `n` giá trị `i` → khoảng `n · (n-1)` phép cộng → **`O(n²)`**.

Với `n = 10^4` → khoảng `10^8` phép cộng: Python sẽ rất chậm (có thể Time Limit Exceeded). Cell dưới đếm số phép cộng thật sự.

In [7]:
def count_additions_bruteforce(n: int) -> int:
    total_ops = 0
    for i in range(n):
        total_ops += i            # cộng bên trái: i phần tử
        total_ops += n - 1 - i    # cộng bên phải
    return total_ops

for n in [10, 100, 1000, 10_000]:
    print(f"n = {n:>6} -> khoảng {count_additions_bruteforce(n):>12,} phép cộng")

n =     10 -> khoảng           90 phép cộng
n =    100 -> khoảng        9,900 phép cộng
n =   1000 -> khoảng      999,000 phép cộng
n =  10000 -> khoảng   99,990,000 phép cộng


## 4. Nút thắt và Insight

**Nút thắt:** khi đi từ `i` sang `i + 1`, tổng bên trái chỉ **thêm đúng một phần tử** `nums[i]`. Vậy mà Brute Force lại cộng lại từ đầu!

```text
left(i+1) = left(i) + nums[i]
```

**Insight 2:** Gọi `total = sum(nums)`. Mọi phần tử nằm ở đúng một trong 3 chỗ: trái, chính `nums[i]`, phải. Nên:

```text
total = left + nums[i] + right
=> right = total - left - nums[i]
```

Điều kiện pivot `left == right` trở thành:

```text
left == total - left - nums[i]
<=> 2 * left + nums[i] == total
```

Ví dụ `nums = [1,7,3,6,5,6]`, `total = 28`. Tại `i = 3`: `left = 11`, `right = 28 - 11 - 6 = 11` ✅.

In [8]:
nums = [1, 7, 3, 6, 5, 6]
total = sum(nums)
left = 1 + 7 + 3
i = 3
right = total - left - nums[i]
print("total =", total, "| left =", left, "| right =", right)
print("2*left + nums[i] =", 2 * left + nums[i], "== total?", 2 * left + nums[i] == total)

total = 28 | left = 11 | right = 11
2*left + nums[i] = 28 == total? True


## 5. Cách 2 — Mảng tổng tiền tố (Prefix Sum array)

Đây là đúng gợi ý (hint) của LeetCode: chuẩn bị trước mảng `prefix` với

```text
prefix[k] = nums[0] + nums[1] + ... + nums[k-1]      (tổng k phần tử đầu)
prefix[0] = 0
```

Khi đó:
- tổng trái của `i` = `prefix[i]`
- tổng phải của `i` = `prefix[n] - prefix[i + 1]`

### 5.1. Xây mảng prefix

`prefix` dài `n + 1`. Mỗi ô được tính từ ô trước: `prefix[k + 1] = prefix[k] + nums[k]`.

In [9]:
nums = [1, 7, 3, 6, 5, 6]
prefix = [0] * (len(nums) + 1)      # [0, 0, ..., 0] dài n + 1
for k in range(len(nums)):
    prefix[k + 1] = prefix[k] + nums[k]
print("nums   =", nums)
print("prefix =", prefix)

nums   = [1, 7, 3, 6, 5, 6]
prefix = [0, 1, 8, 11, 17, 22, 28]


### 5.2. `itertools.accumulate` — thư viện chuẩn tính sẵn tổng chạy

`accumulate(nums)` trả lần lượt các tổng `nums[0]`, `nums[0]+nums[1]`, ... Thêm `initial=0` để có phần tử 0 ở đầu (Python 3.8+).

In [10]:
from itertools import accumulate
print(list(accumulate([1, 7, 3, 6, 5, 6])))
print(list(accumulate([1, 7, 3, 6, 5, 6], initial=0)))

[1, 8, 11, 17, 22, 28]
[0, 1, 8, 11, 17, 22, 28]


In [11]:
def pivot_prefix_array(nums: list[int]) -> int:
    n = len(nums)
    prefix = [0] * (n + 1)
    for k in range(n):
        prefix[k + 1] = prefix[k] + nums[k]
    for i in range(n):
        left = prefix[i]
        right = prefix[n] - prefix[i + 1]
        if left == right:
            return i
    return -1

for nums in [[1, 7, 3, 6, 5, 6], [1, 2, 3], [2, 1, -1]]:
    print(nums, "->", pivot_prefix_array(nums))

[1, 7, 3, 6, 5, 6] -> 3
[1, 2, 3] -> -1
[2, 1, -1] -> 0


Cách 2: thời gian `O(n)` (2 vòng lặp tách rời), bộ nhớ `O(n)` cho mảng `prefix`.

Câu hỏi: ta có thật sự cần **cả mảng** không? Khi duyệt `i` từ trái sang phải, ta chỉ đọc `prefix[i]` đúng một lần rồi bỏ. → Thay mảng bằng **một biến chạy**.

## 6. Cách 3 — Lời giải chính: `total` + biến `left` chạy (O(1) bộ nhớ)

Ghép các mảnh lại:

**Bước 1.** Tính `total = sum(nums)` một lần.

**Bước 2.** Khởi tạo `left = 0` (bên trái của `i = 0` rỗng).

**Bước 3.** Với mỗi `i, x`:
- Tính `right = total - left - x`.
- Nếu `left == right` → trả `i` (là pivot trái nhất vì duyệt từ trái).
- Nếu không: **sau khi kiểm tra** mới cộng `left += x` để chuẩn bị cho `i + 1`.

**Bước 4.** Hết vòng mà không trả → `-1`.

Thứ tự "**kiểm tra trước, cộng sau**" là mấu chốt: lúc kiểm tra `i`, `left` phải chưa chứa `nums[i]`.

In [12]:
def pivot_running_sum(nums: list[int]) -> int:
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        right = total - left - x
        if left == right:
            return i
        left += x
    return -1

for nums in [[1, 7, 3, 6, 5, 6], [1, 2, 3], [2, 1, -1]]:
    print(nums, "->", pivot_running_sum(nums))

[1, 7, 3, 6, 5, 6] -> 3
[1, 2, 3] -> -1
[2, 1, -1] -> 0


## 7. Trace từng bước

Ví dụ `nums = [1, 7, 3, 6, 5, 6]`, `total = 28`.

| i | x | left (trước) | right = 28 - left - x | left == right? | left (sau) |
|---:|---:|---:|---:|:---:|---:|
| 0 | 1 | 0 | 27 | ❌ | 1 |
| 1 | 7 | 1 | 20 | ❌ | 8 |
| 2 | 3 | 8 | 17 | ❌ | 11 |
| 3 | 6 | 11 | 11 | ✅ trả 3 | — |

Ý nghĩa các cột: `left (trước)` là tổng **thực sự** các phần tử trước `i`; `right` được suy ra chứ không cộng lại; cột cuối cho thấy cập nhật xảy ra **sau** khi so sánh.

In [13]:
def trace_pivot(nums: list[int]) -> int:
    total = sum(nums)
    left = 0
    print(f"nums = {nums}, total = {total}")
    for i, x in enumerate(nums):
        right = total - left - x
        ok = left == right
        print(f"i={i} x={x:>3} | left={left:>3} right={right:>3} | {'PIVOT!' if ok else 'chưa'}")
        if ok:
            return i
        left += x
    print("Không có pivot")
    return -1

print("=>", trace_pivot([1, 7, 3, 6, 5, 6]))

nums = [1, 7, 3, 6, 5, 6], total = 28
i=0 x=  1 | left=  0 right= 27 | chưa
i=1 x=  7 | left=  1 right= 20 | chưa
i=2 x=  3 | left=  8 right= 17 | chưa
i=3 x=  6 | left= 11 right= 11 | PIVOT!
=> 3


### Trace có số âm và trường hợp không có pivot

Số âm làm tổng **không tăng đơn điệu**, nên không thể "dừng sớm khi left vượt right". Xem `[2, 1, -1]` (pivot ở đầu) và `[1, 2, 3]` (không có pivot).

In [14]:
print("=>", trace_pivot([2, 1, -1]))
print()
print("=>", trace_pivot([1, 2, 3]))

nums = [2, 1, -1], total = 2
i=0 x=  2 | left=  0 right=  0 | PIVOT!
=> 0

nums = [1, 2, 3], total = 6
i=0 x=  1 | left=  0 right=  5 | chưa
i=1 x=  2 | left=  1 right=  3 | chưa
i=2 x=  3 | left=  3 right=  0 | chưa
Không có pivot
=> -1


### Trace có nhiều pivot — vì sao trả cái trái nhất?

`nums = [0, 0, 0]`: mọi chỉ số đều là pivot (trái = phải = 0). Vì vòng lặp đi từ `i = 0` và `return` ngay khi gặp, kết quả là `0`.

In [15]:
print("=>", trace_pivot([0, 0, 0]))
print("=>", trace_pivot([-1, -1, 0, 1, 1, 0]))

nums = [0, 0, 0], total = 0
i=0 x=  0 | left=  0 right=  0 | PIVOT!
=> 0
nums = [-1, -1, 0, 1, 1, 0], total = 0
i=0 x= -1 | left=  0 right=  1 | chưa
i=1 x= -1 | left= -1 right=  2 | chưa
i=2 x=  0 | left= -2 right=  2 | chưa
i=3 x=  1 | left= -2 right=  1 | chưa
i=4 x=  1 | left= -1 right=  0 | chưa
i=5 x=  0 | left=  0 right=  0 | PIVOT!
=> 5


## 8. Code nộp LeetCode

### Cách 3 (khuyến nghị)

In [16]:
class Solution:
    def pivotIndex(self, nums: list[int]) -> int:
        total = sum(nums)
        left = 0
        for i, x in enumerate(nums):
            if left == total - left - x:
                return i
            left += x
        return -1


sol = Solution()
print(sol.pivotIndex([1, 7, 3, 6, 5, 6]))  # 3
print(sol.pivotIndex([1, 2, 3]))           # -1
print(sol.pivotIndex([2, 1, -1]))          # 0

3
-1
0


### Các phiên bản nộp khác (để so sánh)

`SolutionBruteForce` đúng nhưng `O(n²)` — dễ TLE với `n = 10^4`. `SolutionPrefix` dùng mảng tiền tố. Trên LeetCode, class phải tên `Solution`; ở đây đặt tên khác để cùng tồn tại trong notebook.

In [17]:
class SolutionBruteForce:
    def pivotIndex(self, nums: list[int]) -> int:
        for i in range(len(nums)):
            if sum(nums[:i]) == sum(nums[i + 1:]):
                return i
        return -1


class SolutionPrefix:
    def pivotIndex(self, nums: list[int]) -> int:
        from itertools import accumulate
        prefix = list(accumulate(nums, initial=0))
        n = len(nums)
        for i in range(n):
            if prefix[i] == prefix[n] - prefix[i + 1]:
                return i
        return -1


print(SolutionBruteForce().pivotIndex([1, 7, 3, 6, 5, 6]), SolutionPrefix().pivotIndex([1, 7, 3, 6, 5, 6]))

3 3


## 9. Vì sao lời giải đúng?

**Invariant (bất biến):** ở đầu mỗi vòng lặp với chỉ số `i`, biến `left` bằng đúng `nums[0] + ... + nums[i-1]`.

- *Khởi đầu:* `i = 0`, `left = 0` = tổng của dãy rỗng ✔.
- *Duy trì:* nếu đầu vòng `i` đúng, cuối vòng ta làm `left += nums[i]` → đầu vòng `i + 1`, `left = nums[0] + ... + nums[i]` ✔.

Từ invariant: `right = total - left - nums[i]` đúng bằng tổng các phần tử sau `i` (vì `total` = trái + giữa + phải). Vậy phép so sánh `left == right` kiểm tra **chính xác** định nghĩa pivot.

**Trái nhất:** ta xét `i = 0, 1, 2, ...` theo thứ tự và trả ngay lần đầu điều kiện đúng → không có pivot nào nhỏ hơn bị bỏ qua.

**Trả -1 đúng lúc:** chỉ khi mọi `i` đã kiểm tra và đều sai.

## 10. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute Force (slice + sum) | `O(n²)` | `O(n)` (slice tạm) | dễ nghĩ, chậm với `n = 10^4` |
| 2. Mảng prefix | `O(n)` | `O(n)` | đúng theo hint; tái sử dụng được nhiều truy vấn |
| 3. `total` + `left` chạy | `O(n)` | `O(1)` | ngắn nhất, tối ưu |

Cách 3: `sum(nums)` là 1 lượt `O(n)`, vòng `for` thêm tối đa 1 lượt `O(n)` → tổng `O(n)`. Chỉ dùng vài biến số nguyên → `O(1)`.

Ghi chú: giá trị lớn nhất có thể có của `total` là `10^4 · 1000 = 10^7` — không có vấn đề tràn số (Python số nguyên không giới hạn, và ngay cả `int32` cũng đủ).

## 11. Bộ test

Nhóm test:
1. Ví dụ của đề.
2. Biên: 1 phần tử (luôn là pivot vì trái = phải = 0), pivot ở đầu, pivot ở cuối.
3. Số âm, toàn số 0 (nhiều pivot → trái nhất).
4. Đối chiếu 3 cách trên nhiều input ngẫu nhiên (có seed để lặp lại được).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("đề 1", sol.pivotIndex([1, 7, 3, 6, 5, 6]), 3)
check("đề 2", sol.pivotIndex([1, 2, 3]), -1)
check("đề 3", sol.pivotIndex([2, 1, -1]), 0)
check("1 phần tử", sol.pivotIndex([5]), 0)
check("1 phần tử âm", sol.pivotIndex([-7]), 0)
check("pivot ở cuối", sol.pivotIndex([-1, 1, 5]), 2)
check("toàn 0 -> trái nhất", sol.pivotIndex([0, 0, 0, 0]), 0)
check("số âm", sol.pivotIndex([-1, -1, -1, -1, -1, 0]), 2)
check("hai phần tử", sol.pivotIndex([1, 0]), 0)
check("không pivot 2 phần tử", sol.pivotIndex([1, 1]), -1)

PASS  đề 1: 3
PASS  đề 2: -1
PASS  đề 3: 0
PASS  1 phần tử: 0
PASS  1 phần tử âm: 0
PASS  pivot ở cuối: 2
PASS  toàn 0 -> trái nhất: 0
PASS  số âm: 2
PASS  hai phần tử: 0
PASS  không pivot 2 phần tử: -1


In [19]:
import random

rng = random.Random(724)
methods = [pivot_bruteforce, pivot_prefix_array, pivot_running_sum,
           SolutionBruteForce().pivotIndex, SolutionPrefix().pivotIndex, Solution().pivotIndex]
found_pivot = 0
for _ in range(2000):
    n = rng.randint(1, 12)
    nums = [rng.randint(-3, 3) for _ in range(n)]
    answers = {m(nums) for m in methods}
    assert len(answers) == 1, (nums, answers)
    if answers != {-1}:
        found_pivot += 1
print("2000 input ngẫu nhiên: mọi cách cho cùng kết quả. Số input có pivot:", found_pivot)

2000 input ngẫu nhiên: mọi cách cho cùng kết quả. Số input có pivot: 973


In [20]:
import time

big = [rng.randint(-1000, 1000) for _ in range(10_000)]
for name, f in [("brute force", pivot_bruteforce), ("prefix array", pivot_prefix_array), ("running sum", pivot_running_sum)]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:<13} -> {r:>5}  ({time.perf_counter() - t0:.4f} s)")

brute force   ->    -1  (0.9422 s)
prefix array  ->    -1  (0.0015 s)
running sum   ->    -1  (0.0008 s)


Thời gian cụ thể tùy máy, nhưng Brute Force luôn chậm hơn hẳn (nhất là khi pivot nằm muộn hoặc không có pivot) — đúng với phân tích `O(n²)` so với `O(n)`.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng `x` vào `left` **trước** khi kiểm tra

Khi đó `left` đã chứa `nums[i]`, tức là đang so sánh "trái + giữa" với "phải".

In [21]:
def wrong_update_first(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        left += x                     # SAI: cộng quá sớm
        if left == total - left - x:
            return i
    return -1

print("sai :", wrong_update_first([1, 7, 3, 6, 5, 6]), "| đúng:", pivot_running_sum([1, 7, 3, 6, 5, 6]))

sai : -1 | đúng: 3


**Sửa:** kiểm tra trước, `left += x` sau (như Cách 3).

### Lỗi 2: Quên trừ `nums[i]` khi tính phải: `right = total - left`

Phần tử giữa không thuộc bên nào. Quên trừ thì `right` luôn dư `nums[i]`.

In [22]:
def wrong_forget_middle(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left:      # SAI: thiếu "- x"
            return i
        left += x
    return -1

print("sai :", wrong_forget_middle([1, 7, 3, 6, 5, 6]), "| đúng:", pivot_running_sum([1, 7, 3, 6, 5, 6]))

sai : -1 | đúng: 3


### Lỗi 3: Dùng hai con trỏ / "dừng sớm" như thể mảng toàn số dương

Ý tưởng "đẩy con trỏ phía có tổng nhỏ hơn" dựa trên giả định ngầm rằng tổng chỉ tăng khi thêm phần tử. Đề cho `-1000 <= nums[i]` nên giả định này sai, và kiểu so khớp "gặp nhau ở giữa" rất dễ bỏ sót pivot.

In [23]:
def wrong_two_pointers(nums):
    i, j = 0, len(nums) - 1
    left = right = 0
    while i < j:
        if left + nums[i] <= right + nums[j]:
            left += nums[i]; i += 1
        else:
            right += nums[j]; j -= 1
    return i if left == right else -1

case = [-2, 1, -1]   # pivot đúng là 0: trái 0, phải 1 + (-1) = 0
print("sai :", wrong_two_pointers(case), "| đúng:", pivot_running_sum(case))

sai : -1 | đúng: 0


**Sửa:** dùng `total` + `left` chạy — không phụ thuộc dấu của các số.

### Lỗi 4: Trả về giá trị `nums[i]` thay vì chỉ số `i`, hoặc trả pivot **cuối cùng** (ghi đè biến `ans` rồi trả sau vòng lặp)

Đề hỏi **chỉ số** và **trái nhất** → `return i` ngay trong vòng lặp.

In [24]:
def wrong_last_pivot(nums):
    total, left, ans = sum(nums), 0, -1
    for i, x in enumerate(nums):
        if left == total - left - x:
            ans = i                  # SAI: ghi đè, cuối cùng giữ pivot phải nhất
        left += x
    return ans

print("sai :", wrong_last_pivot([0, 0, 0]), "| đúng:", pivot_running_sum([0, 0, 0]))

sai : 2 | đúng: 0


## 13. Mẹo nhận dạng pattern

Nghĩ ngay tới **Prefix Sum** khi thấy:
- "tổng bên trái / bên phải", "tổng đoạn `[l, r]`", "cân bằng", "chia mảng thành hai phần có tổng ...".
- Cần nhiều lần tính tổng các đoạn **liên tiếp** của cùng một mảng.

Công thức nền tảng cần thuộc:

```text
sum(nums[l:r]) = prefix[r] - prefix[l]          (prefix[k] = tổng k phần tử đầu)
right = total - left - nums[i]                  (chia mảng làm trái / giữa / phải)
```

Khi chỉ duyệt một chiều và mỗi `prefix[i]` dùng đúng 1 lần → thay mảng bằng **một biến chạy** để có `O(1)` bộ nhớ.

## 14. Biến thể

### 14.1. LeetCode 1991 — Find the Middle Index in Array

Đề **giống hệt** bài này (chỉ đổi tên hàm `findMiddleIndex`). Có thể nộp nguyên lời giải.

### 14.2. Trả về **tất cả** pivot

Chỉ cần thay `return i` bằng `append` vào list.

In [25]:
def all_pivots(nums: list[int]) -> list[int]:
    total, left, result = sum(nums), 0, []
    for i, x in enumerate(nums):
        if left == total - left - x:
            result.append(i)
        left += x
    return result

print(all_pivots([0, 0, 0]))            # [0, 1, 2]
print(all_pivots([1, 7, 3, 6, 5, 6]))   # [3]
print(all_pivots([1, -1, 0, 1, -1]))

[0, 1, 2]
[3]
[2]


### 14.3. Chia mảng thành 2 phần (không bỏ phần tử giữa) có tổng bằng nhau

Khác bài gốc: phần tử `i` thuộc bên trái. Điều kiện: `prefix(i+1) == total - prefix(i+1)`, tức tổng tiền tố bằng `total / 2`. Trả về số điểm cắt hợp lệ (cả hai phần khác rỗng).

In [26]:
def count_equal_splits(nums: list[int]) -> int:
    total, run, count = sum(nums), 0, 0
    for x in nums[:-1]:              # điểm cắt sau mỗi phần tử, trừ phần tử cuối
        run += x
        if 2 * run == total:
            count += 1
    return count

print(count_equal_splits([1, 2, 3, 0, 3, 3]))   # 2
print(count_equal_splits([1, 1]))               # 1

2
1


Với `[1, 2, 3, 0, 3, 3]` (`total = 12`), tổng tiền tố bằng 6 tại sau phần tử thứ 3 (`1+2+3`) và sau phần tử thứ 4 (`1+2+3+0`) → 2 cách cắt, khớp output.

### 14.4. Tổng đoạn bất kỳ nhiều lần (LeetCode 303 Range Sum Query)

Xây `prefix` một lần, mỗi truy vấn `sum(l..r)` = `prefix[r+1] - prefix[l]` trong `O(1)`. Đây là lúc **mảng** prefix (Cách 2) hơn hẳn biến chạy.

In [27]:
from itertools import accumulate

class RangeSum:
    def __init__(self, nums):
        self.prefix = list(accumulate(nums, initial=0))
    def query(self, l, r):          # tổng nums[l..r], tính cả hai đầu
        return self.prefix[r + 1] - self.prefix[l]

rs = RangeSum([1, 7, 3, 6, 5, 6])
print(rs.query(0, 2), rs.query(4, 5), rs.query(3, 3))   # 11 11 6

11 11 6


## 15. Bài tập

Làm theo thứ tự, mỗi bài một cell. Cell chỉ **định nghĩa** hàm (có `pass`), test để trong comment — tự bỏ comment khi làm xong.

### Bài 1 — Tính tay
Với `nums = [3, -2, 4, 1, -1, 2]`, lập bảng `i, x, left, right` và cho biết pivot index. (Không cần code; đối chiếu ở phần đáp án.)

### Bài 2 — Xây mảng prefix
Viết `build_prefix(nums)` trả list dài `n + 1` với `prefix[0] = 0`.

In [28]:
def build_prefix(nums: list[int]) -> list[int]:
    # TODO: tạo list [0] * (n + 1) rồi điền prefix[k + 1] = prefix[k] + nums[k]
    pass

# assert build_prefix([1, 7, 3]) == [0, 1, 8, 11]
# assert build_prefix([]) == [0]

### Bài 3 — Điền chỗ trống lời giải chính

In [29]:
def exercise_pivot(nums: list[int]) -> int:
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        # TODO 1: tính right
        # TODO 2: nếu left == right thì trả i
        # TODO 3: cập nhật left (đúng thời điểm!)
        pass
    return -1

# assert exercise_pivot([1, 7, 3, 6, 5, 6]) == 3
# assert exercise_pivot([2, 1, -1]) == 0
# assert exercise_pivot([1, 2, 3]) == -1

### Bài 4 — Pivot **phải nhất**
Trả về chỉ số pivot lớn nhất (không có → `-1`). Yêu cầu `O(n)` thời gian, `O(1)` bộ nhớ và **trả ngay khi tìm thấy** (không duyệt hết rồi lấy cái cuối).

In [30]:
def rightmost_pivot(nums: list[int]) -> int:
    # TODO
    pass

# assert rightmost_pivot([0, 0, 0]) == 2
# assert rightmost_pivot([1, 7, 3, 6, 5, 6]) == 3
# assert rightmost_pivot([1, 2, 3]) == -1

### Bài 5 — Pivot có trọng số
Tìm chỉ số `i` trái nhất sao cho tổng bên trái **bằng 2 lần** tổng bên phải. Không có → `-1`.

In [31]:
def pivot_double(nums: list[int]) -> int:
    # TODO
    pass

# assert pivot_double([4, 1, 2]) == 1     # trái 4, phải 2
# assert pivot_double([1, 1]) == 1        # trái 1, phải 0? -> 1 != 0 ... tự kiểm tra cẩn thận!

### Bài 6 — Mini challenge: Pivot trong ma trận
Cho ma trận `grid` (list các hàng). Tìm chỉ số **hàng** `r` trái nhất (trên nhất) sao cho tổng mọi phần tử các hàng phía trên bằng tổng mọi phần tử các hàng phía dưới. Gợi ý: biến bài 2D thành bài 1D.

In [32]:
def pivot_row(grid: list[list[int]]) -> int:
    # TODO
    pass

# assert pivot_row([[1, 2], [5, 5], [3, 0]]) == 1
# assert pivot_row([[1], [2], [3]]) == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Tính `total` trước | Mỗi hàng: `right = total - left - x` | `total = 7`; dò `i` từ 0 |
| 2 | Mỗi ô dựa vào ô trước | `for k in range(n)` | `prefix[k+1] = prefix[k] + nums[k]` |
| 3 | Kiểm tra trước, cộng sau | `right = total - left - x` | `if left == right: return i` rồi `left += x` |
| 4 | Đối xứng bài gốc | Duyệt ngược, giữ biến `right` | `for i in range(n-1, -1, -1)`: `left = total - right - nums[i]` |
| 5 | Vẫn suy được phải từ trái | Điều kiện mới khác `==` thường | `left == 2 * (total - left - x)` |
| 6 | Mỗi hàng = một số | Tổng từng hàng | `row_sums = [sum(r) for r in grid]` rồi dùng lại lời giải 1D |

Lưu ý bài 5: test thứ hai trong comment cố ý "bẫy" — hãy tự tính lại rồi so với đáp án.

## 17. Đáp án tham khảo

(Chỉ xem sau khi đã tự làm.)

**Bài 1:** `total = 3 - 2 + 4 + 1 - 1 + 2 = 7`.

| i | x | left | right = 7 - left - x | Bằng? |
|---:|---:|---:|---:|:---:|
| 0 | 3 | 0 | 4 | ❌ |
| 1 | -2 | 3 | 6 | ❌ |
| 2 | 4 | 1 | 2 | ❌ |
| 3 | 1 | 5 | 1 | ❌ |
| 4 | -1 | 6 | 2 | ❌ |
| 5 | 2 | 5 | 0 | ❌ |

→ Không có pivot, kết quả `-1` (cell dưới kiểm chứng).

In [33]:
def answer_build_prefix(nums):
    prefix = [0] * (len(nums) + 1)
    for k in range(len(nums)):
        prefix[k + 1] = prefix[k] + nums[k]
    return prefix


def answer_exercise_pivot(nums):
    total, left = sum(nums), 0
    for i, x in enumerate(nums):
        right = total - left - x
        if left == right:
            return i
        left += x
    return -1


def answer_rightmost_pivot(nums):
    total, right = sum(nums), 0
    for i in range(len(nums) - 1, -1, -1):
        left = total - right - nums[i]
        if left == right:
            return i
        right += nums[i]
    return -1


def answer_pivot_double(nums):
    total, left = sum(nums), 0
    for i, x in enumerate(nums):
        if left == 2 * (total - left - x):
            return i
        left += x
    return -1


def answer_pivot_row(grid):
    return answer_exercise_pivot([sum(row) for row in grid])


print("Bài 1:", answer_exercise_pivot([3, -2, 4, 1, -1, 2]))
assert answer_build_prefix([1, 7, 3]) == [0, 1, 8, 11]
assert answer_build_prefix([]) == [0]
assert answer_exercise_pivot([1, 7, 3, 6, 5, 6]) == 3
assert answer_exercise_pivot([2, 1, -1]) == 0
assert answer_exercise_pivot([1, 2, 3]) == -1
assert answer_rightmost_pivot([0, 0, 0]) == 2
assert answer_rightmost_pivot([1, 7, 3, 6, 5, 6]) == 3
assert answer_rightmost_pivot([1, 2, 3]) == -1
assert answer_pivot_double([4, 1, 2]) == 1
print("Bài 5 với [1, 1]:", answer_pivot_double([1, 1]))
assert answer_pivot_row([[1, 2], [5, 5], [3, 0]]) == 1
assert answer_pivot_row([[1], [2], [3]]) == -1
print("Tất cả đáp án PASS")

Bài 1: -1
Bài 5 với [1, 1]: -1
Tất cả đáp án PASS


**Giải thích bài 5 với `[1, 1]`:** tại `i = 0`: trái 0, phải 1 → `0 == 2` sai. Tại `i = 1`: trái 1, phải 0 → `1 == 0` sai. Kết quả `-1`, **không phải** `1` như comment bẫy. Bài học: luôn tính tay test trước khi tin.

**Đối chiếu bài 4 với brute force:**

In [34]:
rng = random.Random(4)
for _ in range(1000):
    nums = [rng.randint(-2, 2) for _ in range(rng.randint(1, 8))]
    brute = [i for i in range(len(nums)) if sum(nums[:i]) == sum(nums[i + 1:])]
    assert answer_rightmost_pivot(nums) == (brute[-1] if brute else -1)
    assert answer_exercise_pivot(nums) == (brute[0] if brute else -1)
print("Bài 3 và Bài 4 khớp brute force trên 1000 input ngẫu nhiên")

Bài 3 và Bài 4 khớp brute force trên 1000 input ngẫu nhiên


## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def pivotIndex(self, nums: list[int]) -> int:
        total = sum(nums)
        left = 0
        for i, x in enumerate(nums):
            if left == total - left - x:   # right = total - left - x
                return i
            left += x                      # cập nhật SAU khi kiểm tra
        return -1
```

**Câu thần chú:** *"total một lần — left chạy — right suy ra — kiểm trước, cộng sau."*

**Checklist trước khi submit:**
- [ ] `left` khởi tạo `0` (bên trái của `i = 0` rỗng)?
- [ ] Công thức phải có **trừ `nums[i]`**: `total - left - nums[i]`?
- [ ] `left += nums[i]` đặt **sau** phép so sánh?
- [ ] `return i` ngay để lấy pivot **trái nhất**, và trả **chỉ số** chứ không phải giá trị?
- [ ] Đã nghĩ tới số âm (không dùng mẹo hai con trỏ / dừng sớm)?
- [ ] Đã test mảng 1 phần tử (kết quả `0`) và mảng không có pivot (`-1`)?